# MNIST demo with CREST 

In [1]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST Datasets

In [2]:
from crest import DataServer

# Load datasets from server
ds_train,ds_valid,ds_test,ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP,OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

2024-01-08 21:10:07.902635: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/draghun1/anaconda3/envs/crest/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Create the HierarchalTensorGraph

In [3]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
import tensorflow as tf
from crest import HierarchalTensorGraph as HTG
from crest.model.TensorSpec import TensorSpec

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})

# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : TensorSpec([None,patch_size, patch_size])},
    outputs={OUT : TensorSpec([None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

{'shape': [None, 28, 28], 'dtype': 'float32', 'name': None}
{'shape': [None, 10], 'dtype': 'float32', 'name': None}
[None, None, None, None, None, None, None]


#### Build a model using the HTG

In [4]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)

# Set build options
build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

/Users/draghun1/Documents/clean/crest/crest/model/Model.py:80: UserWarning: Use Model.compile instead of Model.build
  warnings.warn('Use Model.compile instead of Model.build')


#### Train the model

In [5]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
    'validation_data'  : valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1
    })

Epoch 1/5
250/250 [==============================] - 12s 46ms/step - loss: 4.4786 - sparse_categorical_accuracy: 0.8789 - custom_metric: 27.3637 - val_loss: 0.7405 - val_sparse_categorical_accuracy: 0.9750 - val_custom_metric: 27.4877
Epoch 2/5
250/250 [==============================] - 13s 53ms/step - loss: 0.6545 - sparse_categorical_accuracy: 0.9512 - custom_metric: 27.4056 - val_loss: 0.3723 - val_sparse_categorical_accuracy: 0.9830 - val_custom_metric: 27.4876
Epoch 3/5
250/250 [==============================] - 14s 57ms/step - loss: 0.2662 - sparse_categorical_accuracy: 0.9625 - custom_metric: 27.5007 - val_loss: 0.3128 - val_sparse_categorical_accuracy: 0.9680 - val_custom_metric: 27.4869
Epoch 4/5
250/250 [==============================] - 13s 52ms/step - loss: 0.1651 - sparse_categorical_accuracy: 0.9669 - custom_metric: 27.3948 - val_loss: 0.2143 - val_sparse_categorical_accuracy: 0.9770 - val_custom_metric: 27.4868
Epoch 5/5
250/250 [==============================] - 13s 52m

#### Evaluate the model

In [6]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs= {
    'verbose' : 1,
    'return_dict' : True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - 0s 55ms/step - loss: 0.0621 - sparse_categorical_accuracy: 0.9850 - custom_metric: 26.3266


Accuracy on the full test set should be > ~97%.

### Make predictions

In [7]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False,
    'steps'      : 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
    model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
    print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions

In [8]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_aux = (lat,lon,x)
ds_aux = StructuredDataset(*x_aux, labels=['lat','lon',INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux,coords=['lat','lon'],**pred_kwargs)

{'class': array([[1.1030945e-19, 7.3219197e-20, 1.8615682e-12, ..., 9.9999982e-01,
         6.8446115e-15, 1.3209992e-12],
        [9.5333257e-13, 1.0017401e-12, 9.9999958e-01, ..., 1.6640638e-28,
         2.8211752e-10, 1.2393116e-25],
        [1.0763961e-10, 9.9999994e-01, 8.3241014e-09, ..., 8.9653779e-12,
         2.9470719e-08, 3.6883426e-12],
        ...,
        [9.9345377e-21, 1.4834209e-17, 3.8383021e-18, ..., 2.9857901e-11,
         3.2090248e-09, 1.7299862e-10],
        [4.1278588e-15, 1.6629162e-20, 4.4794897e-20, ..., 4.9311054e-20,
         6.0580554e-09, 5.6490988e-20],
        [6.2745300e-14, 1.2194939e-24, 3.6205973e-17, ..., 2.8420541e-26,
         1.1207913e-18, 1.0441269e-25]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [9]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  100.90108394622803 [s]
